# Training code 

## Import

In [1]:
import aegnn
import pytorch_lightning as pl
import pytorch_lightning.loggers
import wandb
import datetime

from pathlib import Path

import os

/home/kevin-imagine/Documents/event_graph/.venv-gpu/lib/python3.10/site-packages/torchmetrics/utilities/imports.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
os.environ["AEGNN_DATA_DIR"] = "/home/kevin-imagine/Documents/event_graph/data"
print(os.environ["AEGNN_DATA_DIR"])

/home/kevin-imagine/Documents/event_graph/data


## Classification

### Parameters

In [ ]:
params = {"model":"graph_res",
          "task":"",
          "dim":3,
          "seed":12345,
          "lr": 1e-3,
          "weight_decay":5e-3,
          "eta_min":0.0,
          "max_epochs":100,
          "Trainer":{
            "max_epochs":100,
            "overfit_batches":0,
            "log_every_n_steps":10,
            "gradient_clip_val":0,
            "limit_train_batches":1,
            "limit_val_batches":1
          },
          "log-gradients":True,
          "profile":True,
          "debug":True,
          "gpu":0,
          "dataset":"ncars",
          "Data":{
              "batch_size":8,
              "num_workers":8,
              "pin_memory":False
          }          
}

### Model and dataset

In [4]:
data_module = aegnn.datasets.NCars(shuffle = True,
                                   **params["Data"],
                                   )

In [5]:
data_module.setup()

In [6]:
model = aegnn.models.RecognitionModel(params["model"],
                                      params["dataset"],
                                      num_classes=data_module.num_classes,
                                      img_shape=data_module.dims,
                                      dim = params["dim"],
                                      bias = True,
                                      root_weight = True,                                     
                                      )


### Loggers

In [7]:
log_settings = wandb.Settings(start_method='thread')
log_dir = "../training_log_aegnn/"
loggers = [aegnn.utils.loggers.LoggingLogger(None if params["debug"] else log_dir, name="debug")]

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


In [8]:
project = f"aegnn-{params['dataset']}-{params['task']}"
experiment_name = datetime.datetime.now().strftime("%Y%m%d%H%M%S")


In [9]:
if not params["debug"]:
    wandb_logger = pl.loggers.WandbLogger(project=project, save_dir=log_dir, settings=log_settings)
    if params["log-gradients"]:
        wandb_logger.watch(model, log="gradients")
    loggers.append(wandb_logger)
loggers = pl.loggers.LoggerCollection(loggers)

In [10]:
checkpoint_path = os.path.join(log_dir, "checkpoints", params["dataset"], params["task"], experiment_name)
Path(checkpoint_path).mkdir(parents=True,exist_ok=True)

In [11]:
callbacks = [
        pl.callbacks.LearningRateMonitor(),
        aegnn.utils.callbacks.BBoxLogger(classes=data_module.classes),
        # aegnn.utils.callbacks.PHyperLogger(args),
        aegnn.utils.callbacks.EpochLogger(),
        aegnn.utils.callbacks.FileLogger([model, model.model, data_module]),
        aegnn.utils.callbacks.FullModelCheckpoint(dirpath=checkpoint_path)
    ]

In [12]:
trainer_kwargs = {
    "gpus":[params["gpu"]] if params["gpu"] is not None else None,
    "profiler": "simple" if params["profile"] else False,
    "weights_summary": "full",
    "track_grad_norm": 2 if params["log-gradients"] else -1
}

In [13]:
trainer = pl.Trainer(logger=loggers,
                     callbacks=callbacks,
                     **params["Trainer"],
                     **trainer_kwargs
                     )

MisconfigurationException: You requested GPUs: [1]
 But your machine only has: [0]

In [25]:
trainer.fit(model, datamodule=data_module)

je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/training/*
RAW FILer length 12961


100%|██████████| 12961/12961 [00:00<00:00, 99277.14it/s]

je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/validation/*
RAW FILer length 2462



100%|██████████| 2462/2462 [00:00<00:00, 141391.36it/s]
/home/kevin-imagine/Documents/event_graph/.venv-gpu/lib/python3.10/site-packages/pytorch_lightning/core/datamodule.py:423: LightningDeprecationWarning: DataModule.setup has already been called, so it will not be called again. In v1.6 this behavior will change to always call DataModule.setup.
  rank_zero_deprecation(


ValueError: The provided lr scheduler "<torch.optim.lr_scheduler.CosineAnnealingLR object at 0x7d298e8dc190>" is invalid

In [14]:
import torch

In [15]:
torch.cuda.is_available()

True

In [16]:
torch.cuda.get_device_name(0)

'NVIDIA RTX PRO 1000 Blackwell Generation Laptop GPU'